# Этап 1. Первичный аудит исходных данных

Исходные измерения открываются **непосредственно из клона GitHub-репозитория**. Файл читается в `pandas` один раз. Ни одна ячейка не загружает файлы из интерфейса Google Colab.

## Параметры и переменные

- `repo_dir` — каталог клонированного репозитория `ipd`, в котором находится общий модуль анализа.
- `source` — полный путь к конкретному исходному TXT/CSV/TSV/ZIP в репозитории.
- `measurements` — исходные измерения в рабочей таблице `pandas`, где даты уже распознаны, а измерения преобразованы в числа.
- `output_dir` — отдельная папка результата именно этого файла; исходник в ней не изменяется.
- `timestamp` — дата и время соответствующего измерения.
- `Global_active_power` — средняя активная мощность за минуту в кВт.
- `valid_Global_active_power` — признак физически допустимого измерения мощности (`True` — корректно).

Если файлов несколько, раскройте одну строку `source = choose_source("data/raw/ИМЯ.csv")` и укажите файл. Для автоматического анализа всех файлов используйте `python run_analysis.py --all`.

Если у нового файла другие заголовки или мощность указана в ваттах, её единицы и названия задаются в `analysis_config.json` в корне репозитория. Это защищает от ошибочного расчёта энергии.

In [ ]:
# Одна настройка пути к репозиторию. Исходные файлы не загружаются вручную.
from pathlib import Path
import subprocess
import sys

repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ipd_analysis.py").is_file()), None)
if repo_dir is None:
    repo_dir = Path("/content/ipd")
    if not (repo_dir / "ipd_analysis.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sonyxoxo/ipd.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

from ipd_analysis import choose_source, load_measurements, regular_minutes, result_folder, save_csv, save_run_metadata

# Если в data/raw несколько файлов, задайте относительный путь явно:
# source = choose_source("data/raw/household_power_consumption.zip")
source = choose_source()
measurements = load_measurements(source)  # Один pd.read_csv на весь ноутбук.
output_dir = result_folder(source)
output_dir.mkdir(parents=True, exist_ok=True)
print("Источник:", source)
print("Строк:", len(measurements))

## Проверка формы и содержания таблицы

- `audit_table` — одна строка контроля: число строк, начало/конец ряда, повторные и неравномерные временные метки, отсутствующая мощность.
- `sample_rows` — первые десять строк, выбранные лишь для просмотра, а не для анализа вместо полного файла.
- `unknown_values` — словарь количества нечисловых значений по каждому исходному измеряемому столбцу. Значения `?` и пустые ячейки становятся `NaN`, а не нулём.

Контрольные показатели исходного UCI (2 075 259 строк, 25 979 отсутствующих мощностей) полезны для **сравнения**, но не блокируют анализ другого файла.

In [ ]:
from ipd_analysis import audit

audit_table = audit(measurements)
sample_rows = measurements.head(10)
unknown_values = measurements.attrs["unknown_values"]
display(audit_table)
display(sample_rows)
print("Нечисловые обозначения (кроме стандартных пропусков):", unknown_values)
save_csv(audit_table, output_dir / "01_audit.csv")

## Проверка временной сетки

- `minutes` — таблица с **каждой ожидаемой минутой** между началом и концом записи.
- `missing_timestamp_rows` — сколько минут целиком отсутствует в исходном файле. Это **не то же самое**, что строка с `NaN` в мощности.
- `missing_measurements` — сколько минут не имеют пригодной активной мощности после учёта обоих видов пропусков.

Повторяющиеся временные метки сначала нужно исследовать: программа сообщит об ошибке, а не удалит повторы без вашего решения.

In [ ]:
minutes = regular_minutes(measurements)
missing_timestamp_rows = len(minutes) - len(measurements)
missing_measurements = int((~minutes["valid_Global_active_power"]).sum())
print("Пропущенных временных меток:", missing_timestamp_rows)
print("Минут без корректного измерения:", missing_measurements)

## Воспроизводимость результата

Последняя ячейка сохраняет `metadata.json` с SHA-256 исходного файла, хешами исполняемого кода, версиями библиотек и датой запуска. Поэтому любой результат можно связать с конкретным источником и версией обработки.

In [ ]:
save_run_metadata(source, output_dir, {"notebook": 'первичный аудит данных.ipynb'})